# G2: νέα γνώση μόνο στη γειτονιά + ξαναμέτρημα 13β (Colab)
Κανόνες: METRICS.md «Ξαναμέτρημα 13β» και «G2» (commit d1030f6, branch claude/project-thread-tqausr).

**Από τον υπολογιστή, όχι από το κινητό.** Runtime → Change runtime type → **A100**. Το `topic_colab.zip` στο `MyDrive/TopIc/`.

Τρέξε **1 έως 7** με τη σειρά και άφησέ το. Το 1 σταματά αμέσως αν δεν δόθηκε GPU (τότε ξαναδιάλεξε A100).
- 5: ξαναμέτρημα 13β (λίγα λεπτά) → `TopIc/cities_fixed.json`.
- 6: ουρά στο παρασκήνιο: Φάση 1 (6 runs, βάση χωρίς UESP) και μετά Φάση 2 (15 σύντομα runs στο uesp_en).
- 7: μπάρες· στο τέλος γράφει `runs_g2/summary_g2.txt` και **κλείνει την κάρτα**.

Όλα πάνε στο Drive (`TopIc/runs_g2`) με checkpoint. Αν πέσει: ξανατρέχεις 1–7, τα τελειωμένα δεν ξαναξεκινούν.
Ο Φόκους διαβάζει τα αποτελέσματα από το Drive, δεν χρειάζεται να στείλεις τίποτα.

In [ ]:
# 1 · GPU: αλλιώς ΣΤΟΠ (το G1α έτρεξε κατά λάθος σε CPU)
import torch, subprocess
assert torch.cuda.is_available(), '❌ ΔΕΝ ΔΟΘΗΚΕ GPU: Runtime → Change runtime type → A100 και ξανά από εδώ'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ZIP = '/content/drive/MyDrive/TopIc/topic_colab.zip'
assert os.path.exists(ZIP), 'Ανέβασε το topic_colab.zip στο MyDrive/TopIc/'

In [ ]:
%cd /content
!rm -rf Top.Ic && git clone -q -b claude/project-thread-tqausr https://github.com/ppetrakis86-tech/Top.Ic.git
%cd /content/Top.Ic
!git log --oneline -1

In [ ]:
import zipfile, shutil
os.makedirs('corpus_r3', exist_ok=True)
zipfile.ZipFile(ZIP).extractall('corpus_r3')
shutil.move('corpus_r3/r3_data_manifest.json', 'r3_data_manifest.json')
!ls -la corpus_r3
!python -m lightstorm.traps --data-only 16384 --corpus corpus_r3 --manifest r3_data_manifest.json --out gate.json && echo ΠΥΛΗ_OK

In [ ]:
# 5 · ΜΕΡΟΣ Α: ξαναμέτρημα 13β με σωστό σ (ίδια checkpoints, 64 παράθυρα)
T = '/content/drive/MyDrive/TopIc'
OLD = [f'{T}/runs_r9/R100_4', f'{T}/runs_r7/k5_base', f'{T}/runs_r9/R100_16', f'{T}/runs_r8', f'{T}/runs_r9/R25M_16']
print([(r.split('/')[-1], os.path.isdir(r)) for r in OLD])
!python -m lightstorm.cities --runs {' '.join(OLD)} --corpus corpus_r3 --out {T}/cities_fixed.json 2>&1 | tee {T}/cities_fixed.txt

In [ ]:
# 6 · ΜΕΡΟΣ Β: ουρά Φάση 1 → Φάση 2 σε δική της συνεδρία (το ■ δεν την πειράζει). Ξανατρέχοντάς το, συνεχίζει
import json
R = f'{T}/runs_g2'
OLD_SRC = 'wiki_el,wiki_en,greeklish,clean'
src = json.load(open('r3_data_manifest.json'))['sources']
src = {k: v for k, v in src.items() if isinstance(v, dict) and 'train_tokens' in v}
assert {'wiki_el', 'wiki_en', 'greeklish', 'clean', 'uesp_en', 'uesp_el'} <= set(src), sorted(src)
FRAC = sum(src[k]['train_tokens'] for k in OLD_SRC.split(',')) / sum(v['train_tokens'] for v in src.values()) * (1 - 1e-9)
print(f'Φάση 1: {OLD_SRC} · train-frac {FRAC:.6f}')
P1 = [{'out': f'{R}/s{s}', 'arm': arm, 'gb': 3.3,
       'args': ['--epochs', '1', '--eval-every', '1000', '--vocab', '16384', '--corpus', 'corpus_r3', '--manifest', 'r3_data_manifest.json',
                '--deterministic', '--ckpt-every', '1000', '--size', '7M', '--side', '4', '--seed', str(s), '--data-seed', str(s),
                '--sources', OLD_SRC, '--train-frac', f'{FRAC:.9f}']}
      for s in (0, 1, 2) for arm in ('A', 'B')]
ARMS2 = [('A-local', 'A', 'local'), ('B-local', 'B', 'local'), ('B-shuffle', 'B', 'shuffle'), ('A-all', 'A', 'all'), ('B-all', 'B', 'all')]
P2 = [{'out': f'{R}/s{s}/ft', 'arm': name, 'gb': 3.0, 'module': 'lightstorm.finetune', 'done': 'finetune.json',
       'args': ['--ckpt', f'{R}/s{s}/{base}/epoch1.pt', '--mode', mode, '--corpus', 'corpus_r3', '--manifest', 'r3_data_manifest.json',
                '--new', 'uesp_en', '--old', OLD_SRC, '--seed', str(s), '--deterministic']}
      for s in (0, 1, 2) for name, base, mode in ARMS2]
L1, L2 = '/content/qlogs_p1', '/content/qlogs_p2'
os.makedirs(R, exist_ok=True)
for d in (L1, L2, f'{R}/p1', f'{R}/p2'):
    os.makedirs(d, exist_ok=True)
    if os.path.exists(f'{d}/queue_done.json'):
        os.remove(f'{d}/queue_done.json')
json.dump(P1, open(f'{R}/p1/jobs.json', 'w'), indent=1)
json.dump(P2, open(f'{R}/p2/jobs.json', 'w'), indent=1)
cmd = (f'python -m lightstorm.queue_runs --jobs {R}/p1/jobs.json --state {R}/p1 --logs {L1}; '
       f'python -m lightstorm.queue_runs --jobs {R}/p2/jobs.json --state {R}/p2 --logs {L2}')
subprocess.Popen(['bash', '-c', cmd], stdout=open('/content/queue_out_g2.txt', 'a'), stderr=subprocess.STDOUT, start_new_session=True)
print(f'{len(P1)} + {len(P2)} runs στην ουρά → {R}')

In [ ]:
# 7 · ΑΥΤΟΜΑΤΟΣ ΠΙΛΟΤΟΣ: μπάρες (60") → στο τέλος summary_g2.txt στο Drive → κλείνει η κάρτα
import time, re
from IPython.display import clear_output
from lightstorm.queue_runs import run_name
W, t0 = 30, time.time()
def bar(j, logs, total_re=None, fixed=None):
    log, tr = f"{logs}/{run_name(j)}.log", f"{logs}/{run_name(j)}.trace"
    try:
        tot = fixed or int(re.search(total_re, open(log).read()).group(1))
    except (OSError, AttributeError):
        tot = 0
    try:
        st = sum(1 for _ in open(tr))
    except OSError:
        st = 0
    fin = os.path.exists(f"{j['out']}/{j['arm']}/{j.get('done', 'epoch1.json')}")
    if fin:
        st = tot = tot or 1
    n = W * min(st, tot) // tot if tot else 0
    return fin, f"{j['out'][len(R) + 1:] + '/' + j['arm']:20s} {'■' * n}{'·' * (W - n)} {st:5d}/{tot or '?'}" + (' ✅' if fin else '')
while True:
    rows, done_n = [], 0
    for j in P1:
        f, r = bar(j, L1, r'(\d+) steps/εποχή'); done_n += f; rows.append(r)
    for j in P2:
        f, r = bar(j, L2, fixed=1221); done_n += f; rows.append(r)
    clear_output(wait=True)
    print(f'⏱ {(time.time() - t0) / 60:.0f} λεπτά · τελειωμένα {done_n}/{len(P1) + len(P2)}')
    print('\n'.join(rows))
    if os.path.exists(f'{L2}/queue_done.json'):
        break
    alive = int(subprocess.run('ps aux | grep -c "[l]ightstorm.queue_runs"', shell=True, capture_output=True, text=True).stdout or 0)
    if alive == 0 and time.time() - t0 > 600:
        print('⚠️ η ουρά σταμάτησε χωρίς τέλος: γράφω ό,τι υπάρχει και κλείνω')
        break
    time.sleep(60)
from google.colab import runtime
try:
    sh_ = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
    rep = sh_(f'python -m lightstorm.g2report --root {R}')
    rep += '\n--- Φάση 1 (val παλιών πηγών) ---\n' + '\n'.join(
        f"s{s}/{a}: " + (f"{json.load(open(f'{R}/s{s}/{a}/epoch1.json'))['val_loss']:.4f}" if os.path.exists(f'{R}/s{s}/{a}/epoch1.json') else 'ΛΕΙΠΕΙ')
        for s in (0, 1, 2) for a in ('A', 'B'))
    rep += '\n--- ουρές ---\n' + sh_(f'cat {L1}/queue_done.json {L2}/queue_done.json; tail -n 5 {L2}/queue.log; tail -n 10 /content/queue_out_g2.txt')
    open(f'{R}/summary_g2.txt', 'w', encoding='utf-8').write(rep + '\n')
    print(rep)
    print(f'💾 {R}/summary_g2.txt · κλείνω την κάρτα σε 60"')
finally:
    time.sleep(60)
    runtime.unassign()